# LAX — Run the real Streamlit app in Google Colab

Run cells from top to bottom. Choose a GPU runtime (T4 if available). This notebook preserves the PyTorch/TorchVision builds supplied by Colab and starts `app.py` with the v2 adapter stored in Google Drive.


## 1. Mount Drive and confirm the v2 adapter


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
ADAPTER_PATH = Path("/content/drive/MyDrive/psychology-qwen-qlora-v2")
print("Adapter weights exist:", (ADAPTER_PATH / "adapter_model.safetensors").exists())
if not (ADAPTER_PATH / "adapter_model.safetensors").exists():
    raise FileNotFoundError(f"Adapter not found at {ADAPTER_PATH}")


## 2. Clone or update the repository


In [ ]:
%cd /content
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/psychology-raft-rag-chatbot")
if not (REPO_DIR / ".git").exists():
    !git clone https://github.com/karthikeyan-crypto/psychology-raft-rag-chatbot.git
else:
    %cd /content/psychology-raft-rag-chatbot
    !git pull origin main

%cd /content/psychology-raft-rag-chatbot
print("App exists:", Path("app.py").exists())
print("Requirements exists:", Path("requirements.txt").exists())


## 3. Create constraints for Colab's existing PyTorch stack


In [ ]:
import importlib.metadata as metadata
import torch
from pathlib import Path

constraints = [f"torch=={torch.__version__.split('+')[0]}"]

try:
    tv_version = metadata.version("torchvision")
    constraints.append(f"torchvision=={tv_version}")
except metadata.PackageNotFoundError:
    pass

Path("/content/lax-colab-constraints.txt").write_text("\n".join(constraints) + "\n")
print("Keeping Colab package versions:")
print("\n".join(constraints))
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


## 4. Install app dependencies without upgrading Colab's Torch stack


In [ ]:
%cd /content/psychology-raft-rag-chatbot
!python -m pip install -q --prefer-binary -c /content/lax-colab-constraints.txt -r requirements.txt


## Optional compatibility check: TorchAO

Some Colab images include an old optional `torchao` package that Transformers may reject at model-load time. LAX uses bitsandbytes for GPU 4-bit loading, not TorchAO; this cell removes TorchAO only when its version is `0.16.0` or older.

In [ ]:
import importlib.metadata as metadata
from packaging.version import Version
import subprocess, sys

try:
    installed_torchao = metadata.version("torchao")
except metadata.PackageNotFoundError:
    installed_torchao = None

print("TorchAO before check:", installed_torchao)
if installed_torchao is not None and Version(installed_torchao) <= Version("0.16.0"):
    print("Removing incompatible optional TorchAO version; LAX does not need it.")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
else:
    print("No incompatible TorchAO version detected.")


## 5. Verify imports and dependency consistency


In [ ]:
import torch, transformers, peft, bitsandbytes, accelerate, sentence_transformers, faiss, streamlit

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("Accelerate:", accelerate.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("Streamlit:", streamlit.__version__)
!python -m pip check


## 6. Start Streamlit in the background


In [ ]:
!pkill -f "streamlit run app.py" || true


In [ ]:
%cd /content/psychology-raft-rag-chatbot
!nohup streamlit run app.py --server.address=0.0.0.0 --server.port=8501 --server.headless=true >/content/lax.log 2>&1 &


In [ ]:
import time, subprocess
time.sleep(8)
print(subprocess.run(["bash", "-lc", "curl -sS -I http://127.0.0.1:8501"], capture_output=True, text=True).stdout)
print("\n--- Streamlit log ---")
print(subprocess.run(["bash", "-lc", "tail -30 /content/lax.log"], capture_output=True, text=True).stdout)


## 7. Install/cache Cloudflare Tunnel and start the public URL

Leave the final cell running while using LAX. The `trycloudflare.com` URL is temporary and changes when you restart the tunnel.


In [ ]:
from pathlib import Path
TOOLS_DIR = Path("/content/drive/MyDrive/LAX-tools")
TOOLS_DIR.mkdir(parents=True, exist_ok=True)
DEB_PATH = TOOLS_DIR / "cloudflared-linux-amd64.deb"

if not DEB_PATH.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared-linux-amd64.deb
    !cp /content/cloudflared-linux-amd64.deb "$DEB_PATH"

!dpkg -i "$DEB_PATH" >/content/cloudflared-install.log 2>&1 || (cat /content/cloudflared-install.log; exit 1)
!cloudflared --version


In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8501 --protocol http2
